Ячейка 1 — Setup

In [1]:
# %%
"""
SETUP — classmods_parser
"""
import json
import re
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import pandas as pd

BASE = Path("..")
STRUCT = BASE / "data" / "structured_data"
RAW = BASE / "data" / "ncs_raw_json"
DICT_PATH = BASE / "data" / "dictionary" / "result" / "dictionary_en_ru.json"
OUT_DIR = BASE / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("STRUCT:", STRUCT.resolve())
print("DICT:", DICT_PATH.resolve())

STRUCT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/structured_data
DICT: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/dictionary/result/dictionary_en_ru.json


Ячейка 2 — Dictionary

In [2]:
# %%
"""
DICTIONARY — guid → {en, ru}, en_lower → [ru, ...]
"""
with open(DICT_PATH, encoding="utf-8") as f:
    raw_dict = json.load(f)

guid_map = {}
en_map = defaultdict(list)

for k, v in raw_dict.items():
    if not isinstance(v, dict):
        continue
    en = (v.get("en") or "").strip()
    ru = (v.get("ru") or "").strip()
    entry = {"en": en, "ru": ru, "source": v.get("source"), "namespace": v.get("namespace")}
    ku = str(k).replace("-", "").upper()
    guid_map[ku] = entry
    guid_map[str(k).lower()] = entry
    if en:
        en_map[en.lower()].append(ru if ru else None)

def translate_by_guid(guid: str | None) -> str:
    if not guid:
        return "—"
    g = str(guid).replace("-", "").upper()
    hit = guid_map.get(g) or guid_map.get(str(guid).lower())
    if not hit:
        return "(перевод не найден)"
    ru = (hit.get("ru") or "").strip()
    if not ru:
        return "(перевод не найден)"
    return ru

def translate_by_en(text: str | None) -> str:
    if not text or text == "—":
        return "—"
    variants = [r for r in en_map.get(text.lower(), []) if r]
    if not variants:
        return "(перевод не найден)"
    uniq = list(dict.fromkeys(variants))
    if len(uniq) > 1:
        return "(требуется ручная проверка)"
    return uniq[0]

print(f"guid_map: {len(guid_map)} | en_map: {len(en_map)}")

guid_map: 232656 | en_map: 95870


Ячейка 3 — Load structured

In [3]:
# %%
"""
LOAD structured — classmods, bodies, bosses, compositions (для world_drop)
"""
def jload(p: Path):
    with open(p, encoding="utf-8") as f:
        return json.load(f)

cm_items = jload(STRUCT / "classmods" / "all.json")
passives_by_type = jload(STRUCT / "classmods" / "passives_by_type.json")
stats_file = jload(STRUCT / "classmods" / "stats_by_type.json")
bodies_list = jload(STRUCT / "classmods" / "bodies.json")
bosses = jload(STRUCT / "bosses" / "all.json")

stats_g1 = stats_file.get("stat_group1") or {}
stats_g2 = stats_file.get("stat_group2") or {}

# compositions → world_drop / origin
comps: dict[str, dict] = {}
comp_path = STRUCT / "compositions" / "all.json"
if comp_path.exists():
    for c in jload(comp_path):
        comps[(c.get("composition") or "").lower()] = c

body_index = {}
for b in bodies_list:
    it = (b.get("item_type") or "").lower()
    for key in filter(None, [
        (b.get("body_key") or "").lower(),
        (b.get("body_id") or "").lower(),
    ]):
        body_index[(it, key)] = b

handle_to_bosses: dict[str, list] = defaultdict(list)
for b in bosses:
    for h in b.get("dedicated_handles") or []:
        handle_to_bosses[str(h).lower()].append(b)

print(
    f"cm_items={len(cm_items)} | comps={len(comps)} | "
    f"stats_g1_types={len(stats_g1)} | stats_g2_types={len(stats_g2)}"
)
# диагностика статов
for t in list(stats_g1.keys())[:3]:
    print(f"  g1[{t}]={stats_g1.get(t)}")
    print(f"  g2[{t}]={stats_g2.get(t)}")

cm_items=100 | comps=245 | stats_g1_types=1 | stats_g2_types=1
  g1[classmod]=['stat_actionskill_damage', 'stat_all_damage', 'stat_assaultrifle_damage', 'stat_crit_damage', 'stat_criticalhitchance_gun', 'stat_criticalhitchance_melee', 'stat_criticalhitchance_ordnance', 'stat_criticalhitchance_skill', 'stat_damage_reduction', 'stat_elemental_damage', 'stat_fire_rate', 'stat_health_regen', 'stat_kinetic_damage', 'stat_max_health', 'stat_melee_damage', 'stat_ordnance_cooldown', 'stat_ordnance_damage', 'stat_pistol_damage', 'stat_reload_speed', 'stat_shield_capacity', 'stat_shield_regen_rate', 'stat_shotgun_damage', 'stat_skill_cooldown_rate', 'stat_skill_damage', 'stat_skill_duration', 'stat_smg_damage', 'stat_sniper_damage', 'stat_speed', 'stat_splash_damage', 'stat_statuseffect_chance', 'stat_statuseffect_damage', 'stat_weapon_damage', 'statspecial_attunement_duration', 'statspecial_commandskill_cooldown', 'statspecial_cryo_damage', 'statspecial_detonation_damage', 'statspecial_forgedro

Ячейка 4 — Stat labels

In [4]:
# %%
"""
STAT LABELS from uistat / enhancement_uistats in RAW
stat_crit_damage → Critical Hit Damage + guid
"""
GUID_TRIPLE = re.compile(
    r"([A-Za-z0-9_.'/\-]{2,120}),\s*([0-9A-Fa-f]{32}),\s*([^\n\r\"]{2,120})"
)

stat_label: dict[str, dict] = {}  # stat_key → {name_eng, guid}

def _clean_label(s: str) -> str:
    s = re.sub(r"\{[^}]+\}", "", s)
    s = re.sub(r"\[/?[^\]]+\]", "", s)
    s = re.sub(r"\s+", " ", s).strip(" -:\n\t")
    return s

for path in sorted(RAW.rglob("*.json")):
    nl = path.name.lower()
    if not any(x in nl for x in ("uistat", "ui_stat", "tooltip", "gbx_ue")):
        continue
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue
    if "classmod_stat" not in text.lower() and "uistat_classmod_stat" not in text.lower():
        if "stat_" not in text.lower():
            continue
    # key-like uistat_classmod_stat_crit_damage near triple
    for m in re.finditer(r"uistat_classmod_stat_([a-z0-9_]+)", text, re.I):
        sk = "stat_" + m.group(1).lower()
        win = text[m.start(): m.start() + 500]
        tm = GUID_TRIPLE.search(win)
        if not tm:
            continue
        label = _clean_label(tm.group(3))
        if not label or len(label) > 80:
            continue
        if sk not in stat_label:
            stat_label[sk] = {"name_eng": label, "guid": tm.group(2).upper()}

    # also plain "stat_crit_damage" as key in json-ish
    for m in re.finditer(r'"(stat_[a-z0-9_]+)"', text, re.I):
        sk = m.group(1).lower()
        if sk in stat_label:
            continue
        win = text[m.start(): m.start() + 400]
        tm = GUID_TRIPLE.search(win)
        if tm:
            label = _clean_label(tm.group(3))
            if label and len(label) < 80:
                stat_label[sk] = {"name_eng": label, "guid": tm.group(2).upper()}

print(f"stat labels: {len(stat_label)}")
for k, v in sorted(stat_label.items())[:15]:
    print(f"  {k} → {v['name_eng']!r} [{v['guid'][:8]}]")

stat labels: 62
  stat_actionskill_damage → 'Action Skill Damage' [7C828AEB]
  stat_actionskill_damage_and_ordnance_damage → 'Action Skill Damage and Ordnance Damage' [7E7457E0]
  stat_all_damage → 'Damage Dealt' [663B02F2]
  stat_assault_damage → 'Assault Rifle Damage' [86B7575B]
  stat_attunement_duration → 'Attunement Skill Duration' [96EBCCA0]
  stat_commandskill_cooldown → 'Command Skill Cooldown Rate' [208C3137]
  stat_crit_damage → 'Critical Hit Damage' [6AF00314]
  stat_criticalhitchance_gun → 'Gun Critical Hit Chance[/secondary' [51AE0F94]
  stat_criticalhitchance_melee → 'Melee Critical Hit Chance' [F6F46268]
  stat_criticalhitchance_ordnance → 'Ordnance Critical Hit Chance' [2778CB5E]
  stat_criticalhitchance_skill → 'Skill Critical Hit Chance' [551AD2E7]
  stat_cryo_damage → 'Cryo Damage' [FF98A135]
  stat_detonation_damage → 'Detonation Damage' [DEDDA04A]
  stat_dmg_reduction → 'Elemental Damage' [34786B51]
  stat_elem_resist_corrosive → 'Corrosive Resistance' [7E5C1EEE]


Ячейка 5 — Microdicts + helpers

In [5]:
# %%
"""
CELL 5 — MICRODICTS + HELPERS (полная)
source из bosses, passives по body tags, stats с fallback на ключ "classmod",
world_drop из compositions
"""
RARITY_RU = {
    "legendary": "Легендарный",
    "pearlescent": "Перламутровый",
    "epic": "Фиолетовый",
    "rare": "Синий",
    "uncommon": "Зелёный",
    "common": "Белый",
}
CHARACTER_RU = {
    "dark_siren": "Векс",
    "exo_soldier": "Рафа",
    "gravitar": "Харлоу",
    "paladin": "Амон",
    "robodealer": "Н@л",
}

def join_list(xs, sep="; "):
    out = []
    for x in xs:
        if x is None or x == "":
            continue
        s = str(x)
        if s not in out:
            out.append(s)
    return sep.join(out) if out else "—"

def clean_text(s: str | None) -> str:
    if not s:
        return "—"
    s = re.sub(r"\{[^}]*\}", "", s)
    s = re.sub(r"\[/?[^\]]*\]", "", s)
    s = re.sub(r"\s+", " ", s).strip(" -:\n\t")
    return s if s else "—"

def resolve_sources(item: dict) -> dict:
    """
    Только bosses/all.json:
      dedicated_handles → boss_key / display_name / display_guid
    """
    it = (item.get("item_type") or "").lower()
    comp = (item.get("composition") or "").lower()
    body = (item.get("body") or item.get("body_key") or "").lower()
    iid = (item.get("item_id") or "").lower()

    candidates = []
    for c in (
        iid,
        f"{it}.{comp}" if comp else None,
        comp,
        f"{it}.{body}" if body else None,
        body,
    ):
        if c and c not in candidates:
            candidates.append(c)

    matched, seen = [], set()
    for cand in candidates:
        for b in handle_to_bosses.get(cand, []):
            bk = b.get("boss_key")
            if bk not in seen:
                seen.add(bk)
                matched.append(b)

    if not matched:
        for h, blist in handle_to_bosses.items():
            for cand in candidates:
                if cand and (h == cand or h.endswith("." + cand) or h.endswith(cand)):
                    for b in blist:
                        bk = b.get("boss_key")
                        if bk not in seen:
                            seen.add(bk)
                            matched.append(b)

    if not matched:
        return {"source": "—", "source_name_eng": "—", "source_name_ru": "—"}

    tech, eng, rus = [], [], []
    for b in matched:
        tech.append(b.get("boss_key") or "—")
        dn = b.get("display_name")
        dg = b.get("display_guid")
        if dn:
            eng.append(dn)
            rus.append(translate_by_guid(dg) if dg else translate_by_en(dn))
        else:
            eng.append("—")
            rus.append("—")

    return {
        "source": join_list(tech),
        "source_name_eng": join_list(eng),
        "source_name_ru": join_list(rus),
    }

def _body_tags_for_item(item: dict) -> list[str]:
    it = (item.get("item_type") or "").lower()
    body = (item.get("body") or item.get("body_key") or "").lower()
    b = body_index.get((it, body))
    if not b:
        return []
    tags = []
    for t in b.get("addtags") or []:
        tl = str(t).lower()
        if re.match(r"^(red|green|blue)(_left|_right|_mid)?_\d+_\d+$", tl):
            tags.append(tl)
    return tags

def _passive_tag_from_key(pk: str) -> str | None:
    m = re.match(
        r"^passive_((?:red|green|blue)(?:_left|_right|_mid)?_\d+_\d+)(?:_tier_\d+)?$",
        pk.lower(),
    )
    return m.group(1) if m else None

def passives_for_item(item: dict) -> dict:
    """Только навыки этого body (addtags). tier_1 = один скилл на слот."""
    it = (item.get("item_type") or "").lower()
    tags = set(_body_tags_for_item(item))
    rows = passives_by_type.get(it) or []
    keys, names, guids = [], [], []
    seen = set()

    for p in rows:
        if not isinstance(p, dict):
            continue
        pk = (p.get("passive_key") or "").lower()
        if "_tier_" in pk and not pk.endswith("_tier_1"):
            continue
        tag = _passive_tag_from_key(pk)
        if not tags or not tag or tag not in tags:
            continue
        dn = p.get("display_name")
        dg = p.get("display_guid")
        sig = (dg or dn or pk).lower()
        if sig in seen:
            continue
        seen.add(sig)
        keys.append(pk)
        if dn:
            names.append(dn)
            if dg:
                guids.append(dg)

    if not keys:
        return {
            "passive_points": "—",
            "passive_points_name_eng": "—",
            "passive_points_name_ru": "—",
        }

    rus = (
        [translate_by_guid(g) for g in guids]
        if guids
        else [translate_by_en(n) for n in names]
    )
    return {
        "passive_points": join_list(keys),
        "passive_points_name_eng": join_list(names),
        "passive_points_name_ru": join_list(rus) if names else "—",
    }

def _stat_name(k: str) -> tuple[str, str]:
    """(name_eng, name_ru) для одного stat key."""
    kl = k.lower()
    # stat2_crit_damage → пробуем и stat2_ и stat_
    candidates = [kl]
    if kl.startswith("stat2_"):
        candidates.append("stat_" + kl[len("stat2_"):])
    if not kl.startswith("stat"):
        candidates.append("stat_" + kl)

    hit = None
    for sk in candidates:
        hit = stat_label.get(sk)
        if hit:
            break

    if hit:
        eng = clean_text(hit.get("name_eng"))
        ru = clean_text(translate_by_guid(hit.get("guid")))
        if ru in ("(перевод не найден)", "—") and eng not in ("—", ""):
            ru = translate_by_en(eng)
        return eng, ru

    pretty = re.sub(r"^stat2_", "", kl)
    pretty = re.sub(r"^stat(special)?_", "", pretty)
    pretty = pretty.replace("_", " ").strip().title()
    return pretty, translate_by_en(pretty)

def stats_for_item(item: dict) -> dict:
    """
    g1: primary_stats легендарки, иначе пул stat_group1
    g2: пул stat_group2 (ключи stat2_*)
    Structured хранит пулы под ключом "classmod", не classmod_dark_siren.
    """
    it = (item.get("item_type") or "").lower()
    primary = [str(x).lower() for x in (item.get("primary_stats") or [])]

    def pool(group_dict):
        for key in (it, "classmod"):
            v = group_dict.get(key) or group_dict.get(str(key).lower())
            if v:
                return [str(x).lower() for x in v]
        return []

    type_g1 = pool(stats_g1)
    type_g2 = pool(stats_g2)

    if primary:
        g1 = primary
        g2 = type_g2
        if not g2:
            g2 = [x for x in type_g1 if x not in set(primary)]
    else:
        g1 = type_g1
        g2 = type_g2

    def pack(keys, prefix):
        if not keys:
            return {
                prefix: "—",
                f"{prefix}_name_eng": "—",
                f"{prefix}_name_ru": "—",
            }
        names, rus = [], []
        for k in keys:
            e, r = _stat_name(k)
            names.append(e)
            rus.append(r)
        return {
            prefix: join_list(keys),
            f"{prefix}_name_eng": join_list(names),
            f"{prefix}_name_ru": join_list(rus),
        }

    out = {}
    out.update(pack(g1, "stat_group1"))
    out.update(pack(g2, "stat_group2"))
    return out

def part_of_game(item: dict) -> str:
    blob = (
        f"{item.get('item_id', '')} "
        f"{item.get('body', '')} "
        f"{item.get('composition', '')}"
    ).lower()
    found = [
        m
        for m in (
            "raid1",
            "raid2",
            "tuba",
            "cowbell",
            "dlc1",
            "nightmare",
            "cello",
            "banjo",
        )
        if m in blob
    ]
    return join_list(found)

def world_drop_flag(item: dict) -> str:
    """compositions.has_world_drop; body без composition → —."""
    comp = (item.get("composition") or "").lower()
    if not comp:
        return "—"
    c = comps.get(comp)
    if not c:
        return "—"
    return "есть" if c.get("has_world_drop") else "нет"

print("CELL 5 OK | RARITY_RU / helpers ready")
print("stats_g1 classmod:", len(stats_g1.get("classmod") or []))
print("stats_g2 classmod:", len(stats_g2.get("classmod") or []))

CELL 5 OK | RARITY_RU / helpers ready
stats_g1 classmod: 47
stats_g2 classmod: 32


Ячейка 6 — Build rows

In [6]:
# %%
"""
BUILD ROWS
"""
rows = []
src_hit = 0
pas_hit = 0
stat_hit = 0

for it in cm_items:
    rarity = it.get("rarity")
    kind = it.get("kind")

    # legendary + pearlescent + body (фиолетовые имена)
    if kind == "classmod_legendary":
        pass
    elif kind == "classmod_body":
        pass
    elif rarity in ("legendary", "pearlescent", "epic"):
        pass
    else:
        continue

    name_eng = it.get("display_name") or "—"
    if it.get("display_guid"):
        name_ru = translate_by_guid(it.get("display_guid"))
    elif name_eng != "—":
        name_ru = translate_by_en(name_eng)
    else:
        name_ru = "—"

    rar = rarity or ("epic" if kind == "classmod_body" else None)
    rar_ru = RARITY_RU.get(rar, "—") if rar else "—"

    char = it.get("character")
    char_ru = CHARACTER_RU.get(char, char or "—")

    src = resolve_sources(it)
    if src["source"] != "—":
        src_hit += 1

    pas = passives_for_item(it)
    if pas["passive_points_name_eng"] != "—":
        pas_hit += 1

    st = stats_for_item(it)
    if st["stat_group1"] != "—":
        stat_hit += 1

    rows.append({
        "item_id": it.get("item_id") or "—",
        "name_eng": name_eng,
        "name_ru": name_ru,
        "rarity": rar_ru,
        "character": char_ru,
        "body": it.get("body") or it.get("body_key") or "—",
        "world_drop": world_drop_flag(it),
        "part_of_game_eng": part_of_game(it),
        "source": src["source"],
        "source_name_eng": src["source_name_eng"],
        "source_name_ru": src["source_name_ru"],
        **pas,
        **st,
    })

print(f"rows={len(rows)} | with_source={src_hit} | with_passives={pas_hit} | with_stats={stat_hit}")
if rows:
    s = rows[0]
    print("sample item:", s["item_id"], s["name_eng"])
    print("  source:", s["source"], "|", s["source_name_eng"], "|", s["source_name_ru"])
    print("  passives eng:", (s["passive_points_name_eng"] or "")[:120])
    print("  stat1:", s["stat_group1"], s["stat_group1_name_eng"])

rows=100 | with_source=30 | with_passives=100 | with_stats=100
sample item: classmod_dark_siren.comp_05_legendary_01 Technomancer
  source: grasslands_commander | — | —
  passives eng: Restorative Tonic; Full Moon; Ars Arcana; Blast Rites
  stat1: stat_ordnance_damage Ordnance Damage


Ячейка 7 — DataFrame

In [7]:
# %%
"""
DATAFRAME
"""
COLS = [
    "item_id",
    "name_eng",
    "name_ru",
    "rarity",
    "character",
    "body",
    "world_drop",
    "part_of_game_eng",
    "source",
    "source_name_eng",
    "source_name_ru",
    "passive_points",
    "passive_points_name_eng",
    "passive_points_name_ru",
    "stat_group1",
    "stat_group1_name_eng",
    "stat_group1_name_ru",
    "stat_group2",
    "stat_group2_name_eng",
    "stat_group2_name_ru",
]

df = pd.DataFrame(rows)
for c in COLS:
    if c not in df.columns:
        df[c] = "—"
df = df[COLS]
df = df.fillna("—")
print(df.shape)
print(df.head(3).to_string())
print("source non-dash:", int((df["source"] != "—").sum()))
print("passive names non-dash:", int((df["passive_points_name_eng"] != "—").sum()))

(100, 20)
                                    item_id      name_eng      name_ru       rarity character         body world_drop part_of_game_eng                source source_name_eng source_name_ru                                                                                                             passive_points                                                           passive_points_name_eng                                                                    passive_points_name_ru                      stat_group1 stat_group1_name_eng           stat_group1_name_ru                                                                                                                                                                                                                                                                                                                                                                                                                                        

Ячейка 8 — Save CSV

In [8]:
# %%
"""
SAVE bl4_classmods_(date)_(time).csv  — American date/time, Excel-friendly
"""
ts = datetime.now().strftime("%m-%d-%Y_%I-%M-%S%p")
out_path = OUT_DIR / f"bl4_classmods_{ts}.csv"

df.to_csv(
    out_path,
    index=False,
    encoding="utf-8-sig",
    sep=";",
    quoting=1,  # csv.QUOTE_ALL
)

print("Wrote:", out_path.resolve())
print("rows:", len(df))

Wrote: /var/home/nexpg/RawBL4ToCutBDInterpreter/data/output/bl4_classmods_08-17-2026_08-16-48PM.csv
rows: 100
